# Architecture and Prerequisites

Before deploying FAST, understand what it builds and verify your environment is ready.

## FAST Architecture

FAST (Fullstack AgentCore Solution Template) deploys a complete agent application stack using AWS CDK:

| Component | Service | Purpose |
|-----------|---------|---------|
| Frontend | AWS Amplify Hosting | React chat UI with Cognito login |
| Authentication | Amazon Cognito | User pool for frontend login + M2M client for gateway auth |
| Agent Runtime | AgentCore Runtime | Runs the Strands agent in a managed container |
| Model Access | Amazon Bedrock | Foundation model invocation (routed through the LLM Gateway) |
| Tool Execution | AgentCore Gateway | MCP-based tool dispatch to Lambda targets |
| Conversation Memory | AgentCore Memory | Short-term conversation history across turns |
| Code Interpreter | AgentCore Code Interpreter | Secure Python sandbox for calculations |

In this notebook you will:

1. Verify tooling (AWS CLI, CDK, Node, Python, Docker) is present
2. Confirm platform foundations from Modules 2 and 3 are `CREATE_COMPLETE`
3. Confirm the Tools Gateway has travel tools registered

When you finish, you will be ready to clone and deploy FAST in notebook 02.


## Step 1: Verify Tooling

FAST needs Node 18+, CDK v2, Python 3.12+, and Docker (or Finch with `CDK_DOCKER=finch`).


In [ ]:
import shutil, subprocess, sys

REQUIRED = [
    ("aws", ["aws", "--version"]),
    ("node", ["node", "--version"]),
    ("cdk", ["npx", "cdk", "--version"]),
    ("python3", ["python3", "--version"]),
    ("docker", ["docker", "--version"]),
]

missing = []
for name, cmd in REQUIRED:
    if shutil.which(name) is None:
        missing.append(name)
        print(f"  MISSING: {name}")
        continue
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=5).stdout.strip()
        print(f"  {name:8s} {out}")
    except Exception as e:
        print(f"  {name:8s} could not run ({e})")

if missing:
    raise RuntimeError(
        f"Missing CLI tools required by FAST: {missing}. "
        f"Install them before running any cdk deploy cell in notebook 02."
    )

## Step 2: Set Region and Capture Account

FAST (and every notebook in this module) deploys to whatever region your environment is
configured for. The region is resolved from `AWS_REGION`, then `AWS_DEFAULT_REGION`, then your
boto3 session (`aws configure`). Set one of these before running the cells below; the notebook
fails fast if no region is configured.

In [ ]:
import os
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
boto3.setup_default_session(region_name=REGION)

sts = boto3.client("sts", region_name=REGION)
caller = sts.get_caller_identity()
ACCOUNT_ID = caller["Account"]

print(f"Region:     {REGION}")
print(f"Account:    {ACCOUNT_ID}")
print(f"Caller:     {caller['Arn']}")

## Step 3: Verify Platform Foundations

Your platform team deployed the LLM Gateway (Module 2) and the Tool Registry + Tools Gateway
(Module 3a **or** 3b). Both paths produce an AgentCore Gateway that FAST will consume — the
agent code is identical either way.

The next cell probes both paths and reports which one is present.


In [ ]:
cfn = boto3.client("cloudformation", region_name=REGION)

def stack_status(name):
    try:
        return cfn.describe_stacks(StackName=name)["Stacks"][0]["StackStatus"]
    except cfn.exceptions.ClientError:
        return None

STACKS = {
    "workshop-llm-gateway-stack": "Module 2 — LLM Gateway",
    "workshop-registry-stack": "Module 3a — MCP Registry",
    "workshop-tools-gateway-stack": "Module 3a — Tools Gateway",
    "workshop-agentcore-stack": "Module 3b — AgentCore Registry + Gateway",
}

present = {}
for name, label in STACKS.items():
    status = stack_status(name)
    if status is None:
        print(f"  -   {label:42s} {name} (not deployed)")
    else:
        present[name] = status
        marker = "+" if status == "CREATE_COMPLETE" else "!"
        print(f"  {marker}   {label:42s} {name} [{status}]")

llm_ok = present.get("workshop-llm-gateway-stack") == "CREATE_COMPLETE"
mcp_path = (
    present.get("workshop-registry-stack") == "CREATE_COMPLETE"
    and present.get("workshop-tools-gateway-stack") == "CREATE_COMPLETE"
)
agentcore_path = present.get("workshop-agentcore-stack") == "CREATE_COMPLETE"

print()
print(f"LLM Gateway ready:      {llm_ok}")
print(f"MCP path ready (3a):    {mcp_path}")
print(f"AgentCore path (3b):    {agentcore_path}")

if not llm_ok:
    raise RuntimeError("LLM Gateway stack is not CREATE_COMPLETE. Complete Module 2 or wait for pre-deployment.")
if not (mcp_path or agentcore_path):
    raise RuntimeError("Neither Module 3a nor Module 3b is ready. One of them must be CREATE_COMPLETE.")


## Step 4: Verify the Tools Gateway Has Targets

FAST's travel agent needs flights and hotels tools. List the AgentCore gateways in this account
and confirm at least one has travel targets.


In [ ]:
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

gateways = agentcore.list_gateways().get("items", [])
print(f"Gateways found: {len(gateways)}")
for g in gateways:
    print(f"  - {g['name']:30s} id={g['gatewayId']}  status={g.get('status', '?')}")

if not gateways:
    raise RuntimeError("No AgentCore gateways found. Ensure Module 3a or 3b completed.")

preferred = [g for g in gateways if g["name"] in ("tools-gateway", "ac-tools-gateway")]
gw = preferred[0] if preferred else gateways[0]
GATEWAY_ID = gw["gatewayId"]
GATEWAY_NAME = gw["name"]

targets = agentcore.list_gateway_targets(gatewayIdentifier=GATEWAY_ID).get("items", [])
target_names = [t["name"] for t in targets]

print()
print(f"Using gateway:   {GATEWAY_NAME} (id={GATEWAY_ID})")
print(f"Targets ({len(target_names)}):")
for n in target_names:
    print(f"  - {n}")

has_flights = any("flight" in n.lower() for n in target_names)
has_hotels = any("hotel" in n.lower() for n in target_names)
print()
print(f"Flights tool present:   {has_flights}")
print(f"Hotels tool present:    {has_hotels}")

if not (has_flights and has_hotels):
    missing_tools = [t for t, ok in [("flights", has_flights), ("hotels", has_hotels)] if not ok]
    raise RuntimeError(
        f"Gateway '{GATEWAY_NAME}' is missing required targets for FAST: {missing_tools}. "
        f"Complete Module 3a (or 3b) curation before running notebook 02."
    )

## Summary

If the cells above all succeeded, you have:

- AWS CLI, CDK, Node, Python, Docker available locally
- Module 2 LLM Gateway `CREATE_COMPLETE`
- At least one of Module 3a (`workshop-registry-stack` + `workshop-tools-gateway-stack`) or
  Module 3b (`workshop-agentcore-stack`) `CREATE_COMPLETE`
- An AgentCore Gateway with flights and hotels targets

You are ready to proceed to **notebook 02 — Deploy FAST**.
